# 2.2 Matrizen und lineare Gleichungssysteme

In Kapitel 2.1 haben wir das Schwingungssignal eines einzelnen Sensors
modelliert. In der Praxis überwacht man eine Maschine jedoch oft mit mehreren
Sensoren gleichzeitig, zum Beispiel an verschiedenen Lagerstellen. Die
Messwerte aller Sensoren lassen sich dann als **Matrix** organisieren: jede
Zeile entspricht einem Sensor, jede Spalte einem Zeitpunkt. NumPy unterstützt
solche zweidimensionalen Arrays genauso natürlich wie die eindimensionalen
Arrays aus Kapitel 2.1.

Außerdem taucht in der Ingenieurpraxis regelmäßig die Frage auf, welche
Materialeigenschaften oder Kosten hinter einer Reihe von Messungen stecken.
Das führt fast immer auf ein **lineares Gleichungssystem**, das wir mit NumPy
in einer einzigen Zeile lösen können.

## Lernziele

* [ ] Sie können ein zweidimensionales NumPy-Array (Matrix) erzeugen und
  dessen `.shape` interpretieren.
* [ ] Sie können auf einzelne Zeilen, Spalten und Elemente einer Matrix
  zugreifen.
* [ ] Sie können eine **Matrix-Vektor-Multiplikation** mit dem `@`-Operator
  durchführen.
* [ ] Sie können ein lineares Gleichungssystem mit `np.linalg.solve()` lösen.
* [ ] Sie können die Inverse einer Matrix mit `np.linalg.inv()` berechnen.

## Zweidimensionale Arrays

Wir stellen uns vor, dass drei Beschleunigungssensoren unsere Maschine
gleichzeitig überwachen. Jeder Sensor liefert Messwerte zu denselben vier
Zeitpunkten. Diese Daten speichern wir als zweidimensionales Array:

In [ ]:
import numpy as np

# Messwerte: 3 Sensoren, 4 Zeitpunkte (in m/s²)
messungen = np.array([
    [0.3, 1.2, 2.5, 1.8],   # Sensor 1
    [0.5, 0.9, 1.7, 1.1],   # Sensor 2
    [1.2, 2.4, 3.1, 2.0],   # Sensor 3
])

print(messungen)
print(messungen.shape)

`.shape` gibt nun ein Tupel mit zwei Werten zurück: `(3, 4)` bedeutet 3 Zeilen
und 4 Spalten. Die erste Zahl ist immer die Zeilenanzahl, die zweite die
Spaltenanzahl.

**Hinweis: Messdaten anordnen: zeilenweise oder spaltenweise?**
Beide Orientierungen sind in der Praxis üblich. Hier wird *eine Zeile pro
Sensor* gewählt, weil der vollständige Signalverlauf eines Sensors so als
zusammenhängende Zeile vorliegt und gut zur Matrix-Vektor-Multiplikation im
nächsten Abschnitt passt. In der Datenanalyse und im maschinellen Lernen
(pandas, scikit-learn) ist die umgekehrte Anordnung verbreitet: eine Zeile
pro Zeitpunkt, eine Spalte pro Sensor. Entscheidend ist, die gewählte
Konvention konsequent zu verwenden.

Auf einzelne Elemente, Zeilen und Spalten greifen wir mit eckigen Klammern zu:

In [ ]:
print(messungen[0, 2])    # Zeile 0, Spalte 2: Sensor 1 zum dritten Zeitpunkt
print(messungen[1, :])    # Gesamte Zeile 1: alle Werte von Sensor 2
print(messungen[:, 0])    # Gesamte Spalte 0: alle Sensoren zum ersten Zeitpunkt

Der Doppelpunkt `:` bedeutet "alle Einträge in dieser Dimension". Das ist
dasselbe Slicing-Prinzip, das wir von Python-Listen kennen, jetzt aber in
zwei Dimensionen.

**Hinweis: Was steht zuerst: Zeile oder Spalte?**
Bei einem zweidimensionalen Array gilt: Der erste Index wählt die **Zeile**, der
zweite die **Spalte**. `messungen[i, j]` greift auf den Wert in Zeile `i` und
Spalte `j` zu.

### Mini-Übung 1

Gegeben ist folgende Matrix mit Schwingungsamplituden (in mm) von vier
Sensoren an drei verschiedenen Messpunkten:

\begin{equation*}
A = \begin{pmatrix}
0.12 & 0.34 & 0.21 \\
0.45 & 0.11 & 0.38 \\
0.29 & 0.52 & 0.17 \\
0.08 & 0.43 & 0.31 \\
\end{pmatrix}
\end{equation*}

1. Erzeugen Sie die Matrix in Python und geben Sie die Zeilen- und Spaltenanzahl
   aus.
2. Geben Sie die Amplituden von Sensor 3 an allen Messpunkten aus.
3. Geben Sie die Amplituden aller Sensoren am zweiten Messpunkt aus.
4. Was liefert `amplituden[2, 1]`? Ermitteln Sie das Ergebnis zuerst im Kopf.

In [ ]:
# Code-Zelle

## Matrix-Vektor-Multiplikation mit `@`

Eine der wichtigsten Operationen in der linearen Algebra ist die
Matrix-Vektor-Multiplikation. Als Anwendungsbeispiel betrachten wir die
Kalibrierung von Sensoren: Ein Sensor liefert eine elektrische Spannung,
keine physikalische Größe wie Beschleunigung. Die Kalibrierungsmatrix rechnet
die Rohspannungen aller Sensoren in kalibrierte Beschleunigungen um.

In NumPy verwenden wir dafür den `@`-Operator:

In [ ]:
# Kalibrierungsmatrix K (3x3): Umrechnungsfaktoren zwischen Sensoren
K = np.array([
    [2.0, 0.5, 0.0],
    [0.3, 1.8, 0.2],
    [0.0, 0.4, 2.1],
])

# Messvektor: Rohwerte der drei Sensoren zum selben Zeitpunkt (in V)
u = np.array([1.0, 0.8, 1.2])

# kalibrierte Beschleunigungen (in m/s²)
a = K @ u
print(a)

Der `@`-Operator führt die Matrixmultiplikation durch. `K @ u` bedeutet:
multipliziere die Matrix `K` mit dem Vektor `u` und liefere den
Ergebnisvektor `a`. Das Ergebnis hat dieselbe Anzahl von Zeilen wie `K`.

**Hinweis: Wie unterscheiden sich die Multiplikationsoperatoren?**
Der `*`-Operator multipliziert Arrays **elementweise**, der `@`-Operator
führt die **Matrixmultiplikation** durch. `K * u` wäre hier falsch, weil es
nicht der linearen Algebra entspricht. Für zwei Matrizen `A` und `B` gilt:
`A @ B` ist die Matrixmultiplikation, `A * B` ist die elementweise
Multiplikation.

### Mini-Übung 2

Bei einem System aus mehreren Massen, die über Federn verbunden sind, beschreibt
die Steifigkeitsmatrix $K$, welche Kräfte $\vec{F}$ entstehen, wenn die Massen
ausgelenkt werden. Das Produkt $\vec{F} = \mathbf{K}\cdot\vec{x}$ ist die
Matrix-Form des Hookeschen Gesetzes.

Die Steifigkeitsmatrix eines einfachen Feder-Masse-Systems mit der Einheit
N/mm lautet:

\begin{equation*} K = \begin{pmatrix} 3.0 & -1.0\\
    -1.0 &  2.0\\
\end{pmatrix} \end{equation*}

Der Verschiebungsvektor der beiden Massen beträgt $\vec{x} = \begin{pmatrix} 0.5
& 0.3 \end{pmatrix}^\top$ in mm.

1. Berechnen Sie den Kraftvektor $\vec{F} = \mathbf{K}\cdot\vec{x}$ in N.
2. Geben Sie $\vec{F}$ aus und überprüfen Sie das Ergebnis für die erste Komponente
   von Hand: $F_1 = 3.0 \cdot 0.5 + (-1.0) \cdot 0.3$.

In [ ]:
# Code-Zelle

## Lineare Gleichungssysteme lösen mit `np.linalg.solve()`

Wir kehren zu unserer Maschine zurück. Drei Sensoren liefern Messwerte, aber
ihre Kalibrierungsfaktoren sind unbekannt. Aus drei Referenzmessungen mit
bekannten Beschleunigungen lässt sich ein lineares Gleichungssystem aufstellen:

$$\mathbf{A} \cdot \vec{x} = \vec{b}$$

Dabei enthält die Matrix $\mathbf{A}$ die Rohspannungen der Sensoren bei den
Referenzmessungen, $\vec{b}$ die bekannten Referenzbeschleunigungen und
$\vec{x}$ die gesuchten Kalibrierungsfaktoren.

In [ ]:
# Rohspannungen der 3 Sensoren bei 3 Referenzmessungen (in V)
A = np.array([
    [1.0, 0.0, 0.5],
    [0.0, 2.0, 1.0],
    [1.5, 1.0, 0.0],
])

# Bekannte Referenzbeschleunigungen (in m/s²)
b = np.array([2.5, 4.0, 3.0])

# Gesuchte Kalibrierungsfaktoren
x = np.linalg.solve(A, b)
print(f"Kalibrierungsfaktoren: {x}")

`np.linalg.solve(A, b)` löst das Gleichungssystem $\mathbf{A} \cdot \vec{x} =
\vec{b}$ direkt. Intern verwendet NumPy dabei eine sogenannte LU-Zerlegung, die
deutlich effizienter ist als das von Hand durchgeführte Gaußsche
Eliminationsverfahren.

Wir können das Ergebnis überprüfen, indem wir $\mathbf{A} \cdot \vec{x}$
berechnen und mit $\vec{b}$ vergleichen:

In [ ]:
b_probe = A @ x
print(f"Probe A @ x: {b_probe}")
print(f"Original  b: {b}")

Die Werte stimmen überein, das Gleichungssystem ist korrekt gelöst.

## Die Inverse einer Matrix

Eine alternative Methode zum Lösen von $\mathbf{A} \cdot \vec{x} = \vec{b}$
nutzt die Inverse $\mathbf{A}^{-1}$:

$$\vec{x} = \mathbf{A}^{-1} \cdot \vec{b}$$

In NumPy berechnen wir die Inverse mit `np.linalg.inv()`:

In [ ]:
A_inv = np.linalg.inv(A)
x_inv = A_inv @ b
print(f"Lösung via Inverse: {x_inv}")

Das Ergebnis ist dasselbe wie mit `np.linalg.solve()`. In der Praxis ist
`np.linalg.solve()` jedoch vorzuziehen, weil es schneller und numerisch
stabiler ist. Die Berechnung der Inversen ist aufwändiger und anfälliger für
Rundungsfehler. Die Inverse ist dann sinnvoll, wenn man dasselbe
Gleichungssystem für viele verschiedene rechte Seiten $\vec{b}$ lösen möchte,
die Koeffizientenmatrix $\mathbf{A}$ aber immer gleich bleibt.

### Mini-Übung 3

In einer Fabrik werden drei Produkte A, B und C hergestellt. Die
Produktionskosten hängen von den Preisen dreier Materialien ab. Aus drei
bekannten Produkten lässt sich ein lineares Gleichungssystem aufstellen:

$$\begin{pmatrix} 3 & 2 & 1 \\ 0 & 1 & 1 \\ 3 & 1 & 2 \end{pmatrix}
\cdot \begin{pmatrix} M_1 \\ M_2 \\ L \end{pmatrix}
= \begin{pmatrix} 100 \\ 50 \\ 70 \end{pmatrix}$$

$M_1$, $M_2$ sind die Materialpreise in €/Einheit, $L$ ist der Lohnkostensatz
in €/Stunde.

1. Lösen Sie das Gleichungssystem mit `np.linalg.solve()`.
2. Lösen Sie es erneut mit `np.linalg.inv()`.
3. Überprüfen Sie beide Ergebnisse mit einer Probe.

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

Wir haben zweidimensionale NumPy-Arrays als Matrizen kennengelernt und gelernt,
mit dem `@`-Operator Matrixmultiplikationen durchzuführen. Mit
`np.linalg.solve()` lösen wir lineare Gleichungssysteme direkt, ohne das
Gaußsche Eliminationsverfahren von Hand durchzuführen. Die Inverse einer
Matrix lässt sich mit `np.linalg.inv()` berechnen, ist aber in der Regel
nicht die effizienteste Wahl.

Das Schwingungssignal unserer Maschine aus Kapitel 2.1 haben wir in diesem
Kapitel um eine Kalibrierungsanalyse erweitert. Im nächsten Kapitel nutzen wir
NumPy-Werkzeuge wie `np.mean()`, `np.std()` und `np.random`, um das Signal
statistisch zu beschreiben und mit realistischem Messrauschen zu versehen.